# Monte Carlo vs TD vs GAE advantages (CartPole-v1)

This notebook compares **three** policy-gradient setups that share a learned state-value $V_{\phi}(s)$ and REINFORCE-style updates
$$-\sum_t \log \pi_{\theta}(a_t \mid s_t)\,\hat{A}_t.$$
They differ only in how $\hat{A}_t$ is estimated:

1. **Monte Carlo baseline** — $\hat{A}_t = G_t - V_{\phi}(s_t)$ with full discounted returns $G_t$ (same spirit as `reinforce_cartpole_comparison.ipynb` baseline branch).

2. **TD (one-step)** — temporal-difference advantages $\hat{A}_t = \delta_t$ with TD error
$$\delta_t = r_t + \gamma V_{\phi}(s_{t+1})\,(1-\mathrm{done}_t) - V_{\phi}(s_t).$$
This is the **biased, low-variance** extreme (bootstraps on $V$). Algebraically, one-step TD matches $\mathrm{GAE}(\gamma,\lambda)$ with $\lambda=0$.

3. **GAE** — $\mathrm{GAE}(\gamma,\lambda)$ blends TD errors across steps; **`LAMBDA_GAE`** is $\lambda$ (Monte Carlo and TD ignore it).

Prerequisite: REINFORCE basics (`reinforce_cartpole_vanilla.ipynb`) and baseline ideas (`reinforce_cartpole_comparison.ipynb`).


## 1. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Hyperparameters

`LAMBDA_GAE` — **only** for the GAE branch (Schulman et al., $\lambda$ in $\mathrm{GAE}(\gamma,\lambda)$). Try **0.95–0.99** for smoother advantages. **Monte Carlo** and **TD** ignore it.


In [ ]:
ENV_ID = "CartPole-v1"
GAMMA = 0.99
LAMBDA_GAE = 0.95     # λ for GAE(γ, λ); MC and TD branches do not use this
HIDDEN = 128
NUM_EPISODES = 800
PRINT_EVERY = 100

LR_POLICY = 1e-2
LR_VALUE = 2.5e-2

SMOOTH = 50


## 3. Networks

Same **policy** $\pi_{\theta}$ and **value** $V_{\phi}$ for all three runs (matched initialization via seed).


In [ ]:
class Policy(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ValueNet(nn.Module):
    def __init__(self, obs_dim: int, hidden: int = HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


## 4. Returns, TD errors, GAE, and episode updates

**Monte Carlo returns** $G_t$: backward accumulation over the episode (same $G_t$ as in the baseline notebook).

**Monte Carlo advantage:** $\hat{A}_t = G_t - V_{\phi}(s_t)$.

**TD error** (one-step bootstrap with mask $m_t = 1-\mathrm{done}_t$):
$$\delta_t = r_t + \gamma V_{\phi}(s_{t+1})\, m_t - V_{\phi}(s_t).$$
The TD branch uses $\hat{A}_t = \delta_t$.

**GAE:** With the same $\delta_t$, run the backward recurrence (Schulman et al.):
$$\mathrm{GAE}_t = \delta_t + \gamma \lambda\, m_t\, \mathrm{GAE}_{t+1}.$$

**Value targets:** Monte Carlo targets $G_t$ for all branches (fair critic supervision).

**Policy loss:** $-\sum_t \log \pi_{\theta}(a_t \mid s_t)\,\hat{A}_t$ with $\hat{A}$ detached from $\phi$ for the policy gradient (standard practice).


In [ ]:
def discounted_returns_raw(rewards: list[float], gamma: float) -> torch.Tensor:
    G = 0.0
    out: list[float] = []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    out.reverse()
    return torch.tensor(out, dtype=torch.float32, device=device)


def compute_gae(
    rewards: list[float],
    values: torch.Tensor,
    next_values: torch.Tensor,
    dones: list[bool],
    gamma: float,
    lam: float,
) -> torch.Tensor:
    """GAE(γ, λ) advantages; δ uses bootstrap with mask when episode continues."""
    T = len(rewards)
    adv = torch.zeros(T, dtype=torch.float32, device=device)
    gae = torch.zeros((), dtype=torch.float32, device=device)
    for t in reversed(range(T)):
        mask = 1.0 - float(dones[t])
        r_t = torch.tensor(rewards[t], dtype=torch.float32, device=device)
        delta = r_t + gamma * next_values[t] * mask - values[t]
        gae = delta + gamma * lam * mask * gae
        adv[t] = gae
    return adv


def compute_td_advantages(
    rewards: list[float],
    values: torch.Tensor,
    next_values: torch.Tensor,
    dones: list[bool],
    gamma: float,
) -> torch.Tensor:
    """One-step TD advantages A_t = delta_t (same as GAE with lambda=0)."""
    T = len(rewards)
    adv = torch.zeros(T, dtype=torch.float32, device=device)
    for t in range(T):
        mask = 1.0 - float(dones[t])
        r_t = torch.tensor(rewards[t], dtype=torch.float32, device=device)
        delta = r_t + gamma * next_values[t] * mask - values[t]
        adv[t] = delta
    return adv


def run_episode_mc_advantage(
    env: gym.Env,
    policy: Policy,
    value: ValueNet,
    policy_opt: optim.Optimizer,
    value_opt: optim.Optimizer,
    gamma: float,
    train: bool,
) -> float:
    """Monte Carlo advantages A_t = G_t - V(s_t); value targets G_t."""
    log_probs: list[torch.Tensor] = []
    rewards: list[float] = []
    states: list[np.ndarray] = []
    obs, _ = env.reset()
    done = False
    while not done:
        states.append(np.asarray(obs, dtype=np.float32))
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits = policy(obs_t)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(float(reward))
        done = terminated or truncated

    ep_ret = float(sum(rewards))
    if not train or not log_probs:
        return ep_ret

    G = discounted_returns_raw(rewards, gamma)
    states_t = torch.tensor(np.stack(states), dtype=torch.float32, device=device)
    V = value(states_t)
    A = G - V.detach()

    policy_loss = -(torch.stack(log_probs) * A).sum()
    value_loss = F.mse_loss(V, G)

    policy_opt.zero_grad()
    value_opt.zero_grad()
    policy_loss.backward()
    value_loss.backward()
    policy_opt.step()
    value_opt.step()
    return ep_ret


def run_episode_td(
    env: gym.Env,
    policy: Policy,
    value: ValueNet,
    policy_opt: optim.Optimizer,
    value_opt: optim.Optimizer,
    gamma: float,
    train: bool,
) -> float:
    """TD advantages delta_t only; same trajectory layout as GAE."""
    log_probs: list[torch.Tensor] = []
    rewards: list[float] = []
    dones: list[bool] = []
    observations: list[np.ndarray] = []

    obs, _ = env.reset()
    observations.append(np.asarray(obs, dtype=np.float32))
    done = False

    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits = policy(obs_t)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(float(reward))
        done = terminated or truncated
        dones.append(done)
        observations.append(np.asarray(obs, dtype=np.float32))

    ep_ret = float(sum(rewards))
    if not train or not log_probs:
        return ep_ret

    obs_np = np.stack(observations)
    obs_all = torch.tensor(obs_np, dtype=torch.float32, device=device)
    states_t = obs_all[:-1]
    next_states_t = obs_all[1:]
    values = value(states_t)
    next_values = value(next_states_t)

    adv = compute_td_advantages(rewards, values, next_values, dones, gamma)

    G = discounted_returns_raw(rewards, gamma)
    policy_loss = -(torch.stack(log_probs) * adv.detach()).sum()
    value_loss = F.mse_loss(values, G)

    policy_opt.zero_grad()
    value_opt.zero_grad()
    policy_loss.backward()
    value_loss.backward()
    policy_opt.step()
    value_opt.step()
    return ep_ret


def run_episode_gae(
    env: gym.Env,
    policy: Policy,
    value: ValueNet,
    policy_opt: optim.Optimizer,
    value_opt: optim.Optimizer,
    gamma: float,
    lam: float,
    train: bool,
) -> float:
    """GAE advantages; same value targets G_t as MC branch."""
    log_probs: list[torch.Tensor] = []
    rewards: list[float] = []
    dones: list[bool] = []
    observations: list[np.ndarray] = []

    obs, _ = env.reset()
    observations.append(np.asarray(obs, dtype=np.float32))
    done = False

    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits = policy(obs_t)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        obs, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(float(reward))
        done = terminated or truncated
        dones.append(done)
        observations.append(np.asarray(obs, dtype=np.float32))

    ep_ret = float(sum(rewards))
    if not train or not log_probs:
        return ep_ret

    obs_np = np.stack(observations)
    obs_all = torch.tensor(obs_np, dtype=torch.float32, device=device)
    # T steps: states s_0..s_{T-1}, next s_1..s_T
    states_t = obs_all[:-1]
    next_states_t = obs_all[1:]
    values = value(states_t)
    next_values = value(next_states_t)

    adv = compute_gae(rewards, values, next_values, dones, gamma, lam)

    G = discounted_returns_raw(rewards, gamma)
    policy_loss = -(torch.stack(log_probs) * adv.detach()).sum()
    value_loss = F.mse_loss(values, G)

    policy_opt.zero_grad()
    value_opt.zero_grad()
    policy_loss.backward()
    value_loss.backward()
    policy_opt.step()
    value_opt.step()
    return ep_ret


## 5. Train all three (matched initialization)


In [ ]:
def train_mc() -> tuple[list[float], Policy]:
    env = gym.make(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    torch.manual_seed(SEED)
    policy = Policy(obs_dim, n_actions).to(device)
    value = ValueNet(obs_dim).to(device)
    p_opt = optim.Adam(policy.parameters(), lr=LR_POLICY)
    v_opt = optim.Adam(value.parameters(), lr=LR_VALUE)
    rets: list[float] = []
    for ep in range(1, NUM_EPISODES + 1):
        r = run_episode_mc_advantage(env, policy, value, p_opt, v_opt, GAMMA, train=True)
        rets.append(r)
        if ep == 1 or ep % PRINT_EVERY == 0:
            w = rets[-PRINT_EVERY:]
            print(f"[MC advantage] {ep:5d} | return {r:6.1f} | mean {np.mean(w):6.1f}")
    env.close()
    return rets, policy


def train_td() -> tuple[list[float], Policy]:
    env = gym.make(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    torch.manual_seed(SEED)
    policy = Policy(obs_dim, n_actions).to(device)
    value = ValueNet(obs_dim).to(device)
    p_opt = optim.Adam(policy.parameters(), lr=LR_POLICY)
    v_opt = optim.Adam(value.parameters(), lr=LR_VALUE)
    rets: list[float] = []
    for ep in range(1, NUM_EPISODES + 1):
        r = run_episode_td(env, policy, value, p_opt, v_opt, GAMMA, train=True)
        rets.append(r)
        if ep == 1 or ep % PRINT_EVERY == 0:
            w = rets[-PRINT_EVERY:]
            print(f"[TD delta] {ep:5d} | return {r:6.1f} | mean {np.mean(w):6.1f}")
    env.close()
    return rets, policy


def train_gae() -> tuple[list[float], Policy]:
    env = gym.make(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    torch.manual_seed(SEED)
    policy = Policy(obs_dim, n_actions).to(device)
    value = ValueNet(obs_dim).to(device)
    p_opt = optim.Adam(policy.parameters(), lr=LR_POLICY)
    v_opt = optim.Adam(value.parameters(), lr=LR_VALUE)
    rets: list[float] = []
    for ep in range(1, NUM_EPISODES + 1):
        r = run_episode_gae(env, policy, value, p_opt, v_opt, GAMMA, LAMBDA_GAE, train=True)
        rets.append(r)
        if ep == 1 or ep % PRINT_EVERY == 0:
            w = rets[-PRINT_EVERY:]
            print(f"[GAE λ={LAMBDA_GAE}] {ep:5d} | return {r:6.1f} | mean {np.mean(w):6.1f}")
    env.close()
    return rets, policy


returns_mc, policy_mc = train_mc()
returns_td, policy_td = train_td()
returns_gae, policy_gae = train_gae()


## 6. Learning curves


In [ ]:
smooth = np.ones(SMOOTH) / SMOOTH
mc = np.asarray(returns_mc, dtype=np.float64)
td = np.asarray(returns_td, dtype=np.float64)
ret_gae = np.asarray(returns_gae, dtype=np.float64)
ma_x = np.arange(SMOOTH - 1, NUM_EPISODES)

plt.figure(figsize=(9, 4))
plt.plot(mc, alpha=0.2, color="C0", label="MC advantage return")
plt.plot(td, alpha=0.2, color="C2", label="TD (delta) return")
plt.plot(ret_gae, alpha=0.2, color="C1", label=f"GAE (λ={LAMBDA_GAE}) return")
plt.plot(ma_x, np.convolve(mc, smooth, mode="valid"), color="C0", lw=2, label=f"MC MA({SMOOTH})")
plt.plot(ma_x, np.convolve(td, smooth, mode="valid"), color="C2", lw=2, label=f"TD MA({SMOOTH})")
plt.plot(ma_x, np.convolve(ret_gae, smooth, mode="valid"), color="C1", lw=2, label=f"GAE MA({SMOOTH})")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(r"MC $(G_t-V)$ vs TD $\delta_t$ vs GAE — CartPole-v1")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Greedy policy videos (all three runs)

Deterministic $\arg\max_a \pi_{\theta}(a \mid s)$ rollouts for qualitative comparison.


In [ ]:
def record_policy_video(policy: Policy, env_id: str, max_steps: int = 500, seed: int = 123) -> list[np.ndarray]:
    env = gym.make(env_id, render_mode="rgb_array")
    frames: list[np.ndarray] = []
    obs, _ = env.reset(seed=seed)
    done = False
    steps = 0
    policy.eval()
    with torch.no_grad():
        while not done and steps < max_steps:
            frames.append(env.render())
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            logits = policy(obs_t)
            action = int(torch.argmax(logits, dim=1).item())
            obs, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1
    env.close()
    policy.train()
    return frames


def animate_frames(frames: list[np.ndarray], interval_ms: int = 40) -> HTML:
    fig = plt.figure(figsize=(8, 5))
    ax = plt.axes([0, 0, 1, 1], frameon=False)
    ax.axis("off")
    im = ax.imshow(frames[0])

    def update(i: int):
        im.set_data(frames[i])
        return (im,)

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval_ms, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


print("=== Monte Carlo advantages policy ===")
display(animate_frames(record_policy_video(policy_mc, ENV_ID)))

print("=== TD advantages policy ===")
display(animate_frames(record_policy_video(policy_td, ENV_ID)))

print("=== GAE policy ===")
display(animate_frames(record_policy_video(policy_gae, ENV_ID)))


**Note:** TD here uses **one-step** $\delta_t$ as the advantage (same as $\mathrm{GAE}(\gamma,\lambda)$ with $\lambda=0$). Larger $\lambda$ mixes in longer-horizon TD errors. Tune `LAMBDA_GAE` and learning rates if needed.
